<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDS0321ENSkillsNetwork26802033-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Hands-on Lab: Interactive Visual Analytics with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:

*   **TASK 1:** Mark all launch sites on a map
*   **TASK 2:** Mark the success/failed launches for each site on the map
*   **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [2]:
%pip install folium pandas

# Import the libraries
import folium
import pandas as pd
print("Folium and Pandas installed and imported successfully!")

Note: you may need to restart the kernel to use updated packages.
Folium and Pandas installed and imported successfully!


In [3]:
import folium
import pandas as pd

In [4]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/v4/DV0101EN-Exercise-Generating-Maps-in-Python.ipynb)


In [5]:
## Task 1: Mark all launch sites on a map
import pandas as pd
import folium
import requests
import io

# URL for the launch site coordinates
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv'
response = requests.get(URL)
df = pd.read_csv(io.BytesIO(response.content))

# Select relevant columns: LaunchSite, Latitude, Longitude
# Note: In some versions of the lab, a specific 'spacex_launch_geo.csv' is used.
# We will group by LaunchSite to get unique coordinates.
launch_sites_df = df[['LaunchSite', 'Latitude', 'Longitude']].drop_duplicates(subset=['LaunchSite'])
launch_sites_df


,LaunchSite,Latitude,Longitude
0,CCAFS SLC 40,28.561857,-80.577366
3,VAFB SLC 4E,34.632093,-120.610829
26,KSC LC 39A,28.608058,-80.603956


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


In [6]:
import pandas as pd
import requests
import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'

# Use requests instead of js.fetch
response = requests.get(URL)
spacex_csv_file = io.BytesIO(response.content)
spacex_df = pd.read_csv(spacex_csv_file)

# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]

launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Now, you can take a look at what are the coordinates for each site.


In [7]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [ ]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,


In [9]:
import folium
from folium.features import DivIcon

# 1. Initialize the map (centered on a central point among the sites)
nasa_coordinate = [28.562302, -80.577356]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# 2. Iterate through the launch_sites_df to add circles and markers
for index, row in launch_sites_df.iterrows():
    # Define the coordinate for the current site
    site_coord = [row['Lat'], row['Long']]
    site_name = row['Launch Site']
    
    # Create the circle with the popup (following your orange-red color and radius)
    circle = folium.Circle(
        site_coord, 
        radius=1000, 
        color='#d35400', 
        fill=True
    ).add_child(folium.Popup(site_name))
    
    # Create the marker with the text label (following your DivIcon style)
    marker = folium.map.Marker(
        site_coord,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % site_name,
        )
    )
    
    # Add both elements to the map
    site_map.add_child(circle)
    site_map.add_child(marker)

# Display the map
site_map

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.


Now, let's add a circle for each launch site in data frame `launch_sites`


*TODO:*  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


In [10]:
# Initializing the map centered around NASA coordinates
nasa_coordinate = [28.562302, -80.577356]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# For each launch site, add a Circle and a Marker object based on its coordinates
for index, row in launch_sites_df.iterrows():
    # Get site details
    coordinate = [row['Lat'], row['Long']]
    site_name = row['Launch Site']
    
    # 1. Create and add the folium.Circle
    # We use radius=1000 and color=#d35400 to maintain consistency with the label
    circle = folium.Circle(
        coordinate, 
        radius=1000, 
        color='#d35400', 
        fill=True
    ).add_child(folium.Popup(site_name))
    
    # 2. Create and add the folium.Marker with DivIcon
    marker = folium.map.Marker(
        coordinate, 
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0), 
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % site_name, 
        )
    )
    
    # Add children to the map
    site_map.add_child(circle)
    site_map.add_child(marker)

# Display the map
site_map

`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [ ]:
# Initial the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
# For each launch site, add a Circle object based on its coordinate (Lat, Long) values. In addition, add Launch site name as a popup label


In [11]:
# Initialize the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# For each launch site, add a Circle object based on its coordinate (Lat, Long) values. 
# In addition, add Launch site name as a popup label
for index, row in launch_sites_df.iterrows():
    # Define the coordinates for the site
    coordinate = [row['Lat'], row['Long']]
    
    # 1. Add a Circle object for each launch site
    circle = folium.Circle(
        coordinate, 
        radius=1000, 
        color='#d35400', 
        fill=True
    ).add_child(folium.Popup(row['Launch Site']))
    
    # 2. Add a Marker with a DivIcon to display the site name as a text label
    marker = folium.map.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % row['Launch Site'],
        )
    )
    
    # Add the circle and marker to the map
    site_map.add_child(circle)
    site_map.add_child(marker)

# Show the map
site_map

The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:

*   Are all launch sites in proximity to the Equator line?
*   Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


In [13]:
# Task 2: Mark the success/failed launches for each site on the map
from folium.plugins import MarkerCluster

# Function to assign color based on class
# class=1: Success (Green), class=0: Failure (Red)
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'
    
spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)

# Initialize MarkerCluster
marker_cluster = MarkerCluster()

Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [14]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


In [15]:
# Function to assign color based on the value of class column
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'
    
spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [17]:
from folium.plugins import MarkerCluster

# Create a MarkerCluster object
marker_cluster = MarkerCluster()

# Add marker_cluster to the site_map
site_map.add_child(marker_cluster)

In [19]:
marker_cluster = MarkerCluster()


*TODO:* Create a new column in `spacex_df` dataframe called `marker_color` to store the marker colors based on the `class` value


In [ ]:

# Apply a function to check the value of `class` column
# If class=1, marker_color value will be green
# If class=0, marker_color value will be red

*TODO:* For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [20]:
# 1. Create the marker_color column
# We use a lambda function to assign 'green' for success (1) and 'red' for failure (0)
spacex_df['marker_color'] = spacex_df['class'].apply(lambda x: 'green' if x == 1 else 'red')

# 2. Add marker_cluster to current site_map
site_map.add_child(marker_cluster)

# 3. Iterate through spacex_df to create markers
for index, record in spacex_df.iterrows():
    # Create a Marker object with its coordinate [Lat, Long]
    # Customize the Marker's icon property using the marker_color column
    marker = folium.Marker(
        location=[record['Lat'], record['Long']],
        icon=folium.Icon(color='white', icon_color=record['marker_color']),
        popup=f"Launch Site: {record['Launch Site']} | Status: {'Success' if record['class']==1 else 'Failure'}"
    )
    
    # Add the marker to the cluster
    marker_cluster.add_child(marker)

# Display the map
site_map

Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


In [21]:
# TASK 3: Calculate the distances between a launch site to its proximities
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

In [ ]:
# Example: Coordinate of a point on the coastline
# Replace these with actual coordinates found from your map
launch_site_lat = 28.563197
launch_site_lon = -80.576820
coastline_lat = 28.56367
coastline_lon = -80.56763

distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
print(f"Distance to coastline: {distance_coastline:.2f} km")

In [24]:
from math import sin, cos, sqrt, atan2, radians

# 1. Define the distance function (Haversine formula)
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0 # Radius of the earth in km
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

# 2. Set the coordinates (Example for CCAFS LC-40 and nearest coastline)
launch_site_lat = 28.562302
launch_site_lon = -80.577356
coastline_lat = 28.56367
coastline_lon = -80.56763

# 3. Calculate the distance
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

# 4. Now create and add the marker (The code that was failing)
distance_marker = folium.Marker(
   [coastline_lat, coastline_lon],
   icon=DivIcon(
       icon_size=(20,20),
       icon_anchor=(0,0),
       html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline),
       )
   )
site_map.add_child(distance_marker)

# 5. Draw the line between the site and the coastline
lines = folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]], weight=1)
site_map.add_child(lines)

site_map

Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [22]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Next, we need to explore and analyze the proximities of launch sites.
Let's first add a MousePosition on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [25]:
from folium.plugins import MousePosition

# Initialize MousePosition to show coordinates on hover
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

In [26]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


*TODO:* Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [27]:
# find coordinate of the closet coastline
# e.g.,: Lat: 28.56367  Lon: -80.57163
# distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

import pandas as pd
import folium
import requests
import io
from math import sin, cos, sqrt, atan2, radians
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon

# --- STEP 1: HELPER FUNCTIONS & DATA LOADING ---
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0 # Earth radius in km
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon, dlat = lon2 - lon1, lat2 - lat1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    return R * (2 * atan2(sqrt(a), sqrt(1 - a)))

# Load the SpaceX dataset
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
response = requests.get(URL)
spacex_df = pd.read_csv(io.BytesIO(response.content))

# Prepare unique launch sites for the initial markers
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()[['Launch Site', 'Lat', 'Long']]

# --- STEP 2: INITIALIZE MAP ---
nasa_coordinate = [28.562302, -80.577356]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# Add MousePosition for coordinate exploration
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
site_map.add_child(MousePosition(
    position='topright', separator=' Long: ', empty_string='NaN',
    lng_first=False, num_digits=20, prefix='Lat:',
    lat_formatter=formatter, lng_formatter=formatter,
))

# --- STEP 3: ADD LAUNCH SITE MARKERS ---
for index, row in launch_sites_df.iterrows():
    coord = [row['Lat'], row['Long']]
    # Circle for area
    folium.Circle(coord, radius=1000, color='#d35400', fill=True).add_child(folium.Popup(row['Launch Site'])).add_to(site_map)
    # DivIcon for text labels
    folium.map.Marker(coord, icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % row['Launch Site'])).add_to(site_map)

# --- STEP 4: MARKER CLUSTERS (SUCCESS/FAIL) ---
marker_cluster = MarkerCluster().add_to(site_map)
spacex_df['marker_color'] = spacex_df['class'].apply(lambda x: 'green' if x == 1 else 'red')

for index, record in spacex_df.iterrows():
    folium.Marker(
        location=[record['Lat'], record['Long']],
        icon=folium.Icon(color='white', icon_color=record['marker_color']),
        popup=f"Status: {'Success' if record['class']==1 else 'Failure'}"
    ).add_to(marker_cluster)

# --- STEP 5: PROXIMITY DISTANCE (SITE TO COASTLINE) ---
# Example: CCAFS LC-40 to nearest coastline
ls_lat, ls_lon = 28.562302, -80.577356
cl_lat, cl_lon = 28.56367, -80.56763
dist = calculate_distance(ls_lat, ls_lon, cl_lat, cl_lon)

# Add Distance Marker
folium.Marker([cl_lat, cl_lon], icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0),
    html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(dist))).add_to(site_map)

# Add PolyLine
folium.PolyLine(locations=[[ls_lat, ls_lon], [cl_lat, cl_lon]], weight=1).add_to(site_map)

# Final Output
site_map

In [ ]:
# Create and add a folium.Marker on your selected closest coastline point on the map
# Display the distance between coastline point and launch site using the icon property 
# for example
# distance_marker = folium.Marker(
#    coordinate,
#    icon=DivIcon(
#        icon_size=(20,20),
#        icon_anchor=(0,0),
#        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance),
#        )
#    )

In [28]:
# 1. Define the coastline coordinate (found via MousePosition)
coastline_lat = 28.56367
coastline_lon = -80.56763
coastline_coordinate = [coastline_lat, coastline_lon]

# 2. Calculate the distance (ensure launch_site_lat/lon are defined from your site of interest)
# Example using CCAFS LC-40
launch_site_lat = 28.562302
launch_site_lon = -80.577356

distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

# 3. Create and add the folium.Marker
distance_marker = folium.Marker(
    coastline_coordinate,
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline),
    )
)

# 4. Add the marker to the map
site_map.add_child(distance_marker)

# 5. Draw a PolyLine between launch site and coastline for visual clarity
lines = folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], coastline_coordinate], weight=1)
site_map.add_child(lines)

# Display map
site_map

<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


*TODO:* Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


*TODO:* Draw a `PolyLine` between a launch site to the selected coastline point


Your updated map with distance line should look like the following screenshot:

# Create a `folium.PolyLine` object using the coastline coordinates and launch site coordinate
# lines=folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)

In [29]:
# 1. Define the coordinates for the line (Launch Site and Coastline)
# Using coordinates for CCAFS LC-40 and the nearest coastline point
coordinates = [[28.562302, -80.577356], [28.56367, -80.56763]]

# 2. Create the PolyLine object
# we set weight=1 for a thin, clean line
lines = folium.PolyLine(locations=coordinates, weight=1)

# 3. Add the line to the map
site_map.add_child(lines)

# Display the map to see the connection
site_map

In [ ]:
# Create a marker with distance to a closest city, railway, highway, etc.
# Draw a line between the marker to the launch site


In [30]:
# 1. Define the coordinates for the launch site and proximities
# Launch Site: CCAFS LC-40
ls_lat, ls_lon = 28.562302, -80.577356

# Proximities (Coordinates found via MousePosition)
proximities = {
    'Coastline': [28.56367, -80.56763],
    'Highway': [28.56321, -80.57079], # Samuel C Phillips Pkwy
    'Railway': [28.57205, -80.58527], # NASA Railroad
    'City': [28.61200, -80.80700]      # Titusville
}

# 2. Loop through the proximities to calculate distance and add map elements
for name, coord in proximities.items():
    # Calculate distance
    dist = calculate_distance(ls_lat, ls_lon, coord[0], coord[1])
    
    # Create and add the Distance Marker
    distance_marker = folium.Marker(
        coord,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html=f'<div style="font-size: 12px; color:#d35400;"><b>{dist:10.2f} KM</b></div>',
        )
    )
    site_map.add_child(distance_marker)
    
    # Create and add the PolyLine
    line = folium.PolyLine(locations=[[ls_lat, ls_lon], coord], weight=1)
    site_map.add_child(line)

# Display the map
site_map

After you plot distance lines to the proximities, you can answer the following questions easily:

*   Are launch sites in close proximity to railways?
*   Are launch sites in close proximity to highways?
*   Are launch sites in close proximity to coastline?
*   Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


Proximity Findings
Are launch sites in close proximity to railways?
Yes. Launch sites are typically located very close to specialized railroad networks.

Are launch sites in close proximity to highways?
Yes. They are usually situated near major access roads or highways.

Are launch sites in close proximity to coastline?
Yes. The distance to the coastline is usually minimal (often less than 2 km).

Do launch sites keep certain distance away from cities?
Yes. There is a significant buffer zone (usually 20 km or more) between the launch pad and the nearest densely populated city.

Explanation of FindingsThe positioning of SpaceX launch sites is not random; it is a calculated balance of logistics, safety, and physics.1. Logistics: The Need for InfrastructureRockets like the Falcon 9 are massive and heavy. They cannot be transported via standard commercial flight or simple trucks over long distances.Railways: Essential for moving heavy rocket stages and large propellant tanks from manufacturing facilities or storage to the launch site.Highways: Necessary for the daily transport of crew, smaller equipment, and technical support vehicles.2. Safety: The Coastline StrategyThe most critical safety feature of any launch site is its proximity to the ocean.Flight Path: Rockets are launched over the water. If a malfunction occurs during the initial stages of flight (an "anomaly"), the debris falls into the ocean rather than onto residential areas or critical land infrastructure.Stage Recovery: For SpaceX, the coastline is vital for landing the first stage on a droneship located out at sea.3. Public Safety: Buffer from CitiesThe large distance from cities (like Titusville near KSC or Lompoc near VAFB) serves two main purposes:Noise and Vibration: The acoustic energy generated during a launch is powerful enough to shatter windows and cause structural damage if the pad were too close to a city.Risk Mitigation: In the highly unlikely event of a catastrophic failure on the pad, the distance ensures that the explosion's pressure wave and toxic fumes do not reach large populations.4. Physics: Earth's RotationWhile not a proximity to a landmark, sites like Cape Canaveral are chosen because they are closer to the Equator. Launching closer to the equator allows the rocket to use the Earth's natural rotational speed ($1,600\text{ km/h}$) as a "boost," saving significant amounts of fuel.

# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


In [32]:
import pandas as pd
import dash
from dash import html, dcc
from dash.dependencies import Input, Output
import plotly.express as px

# Load the data
spacex_df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv")
max_payload = spacex_df['Payload Mass (kg)'].max()
min_payload = spacex_df['Payload Mass (kg)'].min()

# Initialize the Dash app
app = dash.Dash(__name__)

# Create the app layout
app.layout = html.Div(children=[
    html.H1('SpaceX Launch Records Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 40}),
    
    # TASK 1: Dropdown list to enable Launch Site selection
    dcc.Dropdown(id='site-dropdown',
                options=[
                    {'label': 'All Sites', 'value': 'ALL'},
                    {'label': 'CCAFS LC-40', 'value': 'CCAFS LC-40'},
                    {'label': 'VAFB SLC-4E', 'value': 'VAFB SLC-4E'},
                    {'label': 'KSC LC-39A', 'value': 'KSC LC-39A'},
                    {'label': 'CCAFS SLC-40', 'value': 'CCAFS SLC-40'},
                ],
                value='ALL',
                placeholder="Select a Launch Site here",
                searchable=True
                ),
    html.Br(),

    # TASK 2: Add a pie chart to show the total successful launches count for all sites
    html.Div(dcc.Graph(id='success-pie-chart')),
    html.Br(),

    html.P("Payload range (Kg):"),
    # TASK 3: Add a slider to select payload range
    dcc.RangeSlider(id='payload-slider',
                    min=0, max=10000, step=1000,
                    marks={0: '0', 2500: '2500', 5000: '5000', 7500: '7500', 10000: '10000'},
                    value=[min_payload, max_payload]),

    # TASK 4: Add a scatter chart to show the correlation between payload and launch success
    html.Div(dcc.Graph(id='success-payload-scatter-chart')),
])

# TASK 2: Callback for Pie Chart
@app.callback(Output(component_id='success-pie-chart', component_property='figure'),
              Input(component_id='site-dropdown', component_property='value'))
def get_pie_chart(entered_site):
    filtered_df = spacex_df
    if entered_site == 'ALL':
        fig = px.pie(filtered_df, values='class', 
        names='Launch Site', 
        title='Total Success Launches By Site')
        return fig
    else:
        # Filter for specific site and count success (1) vs failure (0)
        df_site = spacex_df[spacex_df['Launch Site'] == entered_site]
        df_site = df_site.groupby(['class']).size().reset_index(name='class count')
        fig = px.pie(df_site, values='class count', 
        names='class', 
        title=f'Total Success Launches for site {entered_site}')
        return fig

# TASK 4: Callback for Scatter Chart
@app.callback(Output(component_id='success-payload-scatter-chart', component_property='figure'),
              [Input(component_id='site-dropdown', component_property='value'), 
               Input(component_id='payload-slider', component_property='value')])
def get_scatter_chart(entered_site, payload_range):
    low, high = payload_range
    mask = (spacex_df['Payload Mass (kg)'] > low) & (spacex_df['Payload Mass (kg)'] < high)
    filtered_df = spacex_df[mask]
    
    if entered_site == 'ALL':
        fig = px.scatter(filtered_df, x='Payload Mass (kg)', y='class', 
                         color="Booster Version Category",
                         title='Correlation between Payload and Success for all Sites')
        return fig
    else:
        df_site = filtered_df[filtered_df['Launch Site'] == entered_site]
        fig = px.scatter(df_site, x='Payload Mass (kg)', y='class', 
                         color="Booster Version Category",
                         title=f'Correlation between Payload and Success for site {entered_site}')
        return fig


# Run the app
if __name__ == '__main__':
    app.run(debug=True)

---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
Cell In[32], line 79, in get_scatter_chart(
    entered_site='ALL',
    payload_range=[0, 9600]
)
     76 filtered_df = spacex_df[mask]
     78 if entered_site == 'ALL':
---> 79     fig = px.scatter(filtered_df, x='Payload Mass (kg)', y='class', 
        filtered_df =     Flight Number        Date Time (UTC) Booster Version   Launch Site  \
2               3  2012-05-22    7:44:00  F9 v1.0  B0005   CCAFS LC-40   
3               4  2012-10-08    0:35:00  F9 v1.0  B0006   CCAFS LC-40   
4               5  2013-03-01   15:10:00  F9 v1.0  B0007   CCAFS LC-40   
5               7  2013-12-03   22:41:00         F9 v1.1   CCAFS LC-40   
6               8  2014-01-06   22:06:00         F9 v1.1   CCAFS LC-40   
7               9  2014-04-18   19:25:00         F9 v1.1   CCAFS LC-40   
8              10  2014-07-14   15:15:00         F9 v1.1   CC

## Authors


Capt. L K Tauhidur Rahman (Retd)

<!--## Change Log--!>


<!--| Date (YYYY-MM-DD) | Version | Changed By      | Change Description      |
| ----------------- | ------- | -------------   | ----------------------- |
| 2022-11-09        | 1.0     | Pratiksha Verma | Converted initial version to Jupyterlite|--!>


### <h3 align="center"> IBM Corporation 2022. All rights reserved. <h3/>
